# MMDP-unify: A Unified Framework for Multimodal Depression Prediction with Hierarchical Feature Extraction and Dynamic Modal Configuration

---

## Framework Overview

This notebook provides the **complete model implementation** for the IEMDDM framework,  
covering three benchmark datasets with unified architecture and adaptive modal configuration:

| Dataset | Modality Config | Fusion Strategy |
|---------|----------------|-----------------|
| **D-Vlog**  | Audio + Video (bimodal) | Cross-Modal Attention **A** + Gating **g** |
| **EATD**  | Audio + Text (bimodal) | Cross-Modal Attention **A** + Gating **g** (with Word-Level Forced Alignment) |
| **CMDC** | Audio + Text + Video (trimodal) | Hierarchical **G(A,T) → CA(·,V)** |

### Key Modules:
1. **IntraModalFusion** — Transformer self-attention encoder (LinearProj → PosEnc → MultiHeadAttn → FFN → GlobalAvgPool)
2. **bert-base-chinese** — Text encoder with 768→256 linear projection (EATD / CMDC)
3. **Vision Transformer + Pose Estimation** — Dual-path video encoder with Gated Residual + Multi-view Gating (CMDC)
4. **Cross-Modal Attention A + Adaptive Gating g** — Bimodal fusion
5. **Hierarchical G(A,T)→CA(·,V) + Tri-Modal Gate** — Trimodal fusion
6. **Classifier** — 512→256→128→2 MLP with LayerNorm + Dropout
7. **Weighted Cross-Entropy Loss** — Inverse frequency weighting for class imbalance

## 1. Environment Setup & Imports

In [ ]:
import os
import math
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Optional: Uncomment if using text/video encoders
# from transformers import BertTokenizer, BertModel

# ===================== Global Configuration =====================
SEED = 42

def set_seed(seed=SEED):
    """Fix random seeds for reproducibility (PyTorch, NumPy, CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## 2. Hyperparameter Configuration (Table 4-1)

All hyperparameters are defined here and shared across datasets.  
Dataset-specific settings are toggled by the `DatasetConfig` dataclass.

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    """
    IEMDDM Unified Hyperparameter Configuration.
    Corresponds to Table 4-1 in the paper.
    """
    # === Model Architecture ===
    d_model: int = 256              # Unified embedding dimension for all modalities
    num_heads: int = 8              # Multi-head attention heads
    ffn_dim: int = 1024             # FFN inner dimension = 4 * d_model
    dropout: float = 0.3            # Dropout rate applied after each sub-layer
    num_encoder_layers: int = 1     # Number of Transformer encoder layers per modality

    # === Training ===
    learning_rate: float = 1e-4     # Initial LR with cosine annealing
    weight_decay: float = 1e-4      # AdamW weight decay
    batch_size: int = 16
    max_epochs: int = 50
    patience: int = 10              # Early stopping patience

    # === Classifier Head ===
    # 512 → 256 → 128 → 2 (with LayerNorm + Dropout between layers)
    classifier_dims: tuple = (512, 256, 128, 2)

config = Config()
print(config)

Config(d_model=256, num_heads=8, ffn_dim=1024, dropout=0.3, num_encoder_layers=1, learning_rate=0.0001, weight_decay=0.0001, batch_size=16, max_epochs=50, patience=10, classifier_dims=(512, 256, 128, 2))


## 3. Core Model Components

### 3.1 Positional Encoding (Sinusoidal)

Standard sinusoidal positional encoding (Vaswani et al., 2017) to inject  
absolute temporal information into sequences before self-attention.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding (Eq. in Section 3.3.1).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            (batch, seq_len, d_model) with positional encoding added
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

### 3.2 IntraModalFusion Encoder (Section 3.3)

The core intra-modal feature encoder shared by audio, text, and video modalities.  
Architecture: **LinearProj → PosEnc → MultiHeadAttn → FFN → GlobalAvgPool**

- Linear projection maps raw features to `d_model=256`
- Multi-head self-attention with 8 heads (d_k=32)
- FFN with GELU activation: `d_model → 4*d_model → d_model`
- Add & LayerNorm residual connections
- Global Average Pooling for fixed-length output

In [ ]:
class IntraModalFusion(nn.Module):
    """
    Intra-Modal Feature Encoder based on Transformer Self-Attention.
    (Section 3.3: LinearProj → PosEnc → MultiHeadAttn → FFN → GlobalAvgPool)

    Shared architecture across audio, text (post-BERT), and video modalities.
    Ensures all modalities are projected to a unified d_model=256 space.

    Args:
        input_dim:   Raw feature dimension (e.g., 25 for D-Vlog audio, 136 for D-Vlog video)
        d_model:     Unified embedding dimension (default=256)
        num_heads:   Number of attention heads (default=8, d_k=32)
        dropout:     Dropout rate (default=0.3)
        num_layers:  Number of stacked encoder layers (default=1)
    """
    def __init__(self, input_dim: int, d_model: int = 256, num_heads: int = 8,
                 dropout: float = 0.3, num_layers: int = 1):
        super().__init__()

        # --- Linear Projection (Eq. 3.2) ---
        # Maps raw input features to d_model-dimensional embedding space
        self.linear_proj = nn.Linear(input_dim, d_model)

        # --- Positional Encoding ---
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # --- Transformer Encoder ---
        # MultiHeadAttn + FFN (dim_feedforward = 4 * d_model = 1024, GELU activation)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,  # 1024
            dropout=dropout,
            activation='gelu',            # GELU for smoother gradients (Section 3.3.2)
            batch_first=True,
            norm_first=False              # Post-LN (Add & LayerNorm)
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        # LayerNorm after projection
        self.proj_norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        """
        Args:
            x:    (batch, seq_len, input_dim)  — raw feature sequence
            mask: optional padding mask
        Returns:
            seq_out: (batch, seq_len, d_model) — encoded sequence
            pooled:  (batch, d_model)          — GlobalAvgPool output (Eq. 3.4)
        """
        # Linear Projection + LayerNorm
        h = self.linear_proj(x)          # (B, T, d_model)
        h = self.proj_norm(h)

        # Positional Encoding
        h = self.pos_encoder(h)          # (B, T, d_model)

        # Multi-Head Self-Attention + FFN (with Add & LayerNorm)
        seq_out = self.transformer_encoder(h, src_key_padding_mask=mask)  # (B, T, d_model)

        # Global Average Pooling (Section 3.3.2)
        # Robust against outlier frames; better generalization on small datasets
        if mask is not None:
            # Zero out padded positions before averaging
            mask_expanded = (~mask).unsqueeze(-1).float()  # (B, T, 1)
            pooled = (seq_out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        else:
            pooled = seq_out.mean(dim=1)  # (B, d_model)

        return seq_out, pooled

### 3.3 Text Encoder: bert-base-chinese (Section 3.4)

For EATD and CMDC (Chinese datasets), text is encoded by `bert-base-chinese`.  
The 768-dim output is projected to `d_model=256` via a linear layer + LayerNorm.

**Layer-wise Learning Rate Decay**: Layers 1-6 use 1/10 of the top-layer LR.

In [ ]:
class TextEncoder(nn.Module):
    """
    Text Feature Extractor using bert-base-chinese (Section 3.4).

    Pipeline:
        PLM([CLS], w1, ..., wN, [SEP]) → 768-dim → Linear(768→256) → LayerNorm → h_text

    Activated for: EATD, CMDC (Chinese datasets)
    Not activated for: D-Vlog (audio + video only)

    Args:
        pretrained_model: HuggingFace model name (default='bert-base-chinese')
        d_model:          Target embedding dimension (default=256)
        freeze_layers:    Number of bottom BERT layers to freeze (for layer-wise LR decay)
    """
    def __init__(self, pretrained_model: str = 'bert-base-chinese',
                 d_model: int = 256, freeze_layers: int = 0):
        super().__init__()
        from transformers import BertModel

        self.bert = BertModel.from_pretrained(pretrained_model)
        bert_dim = self.bert.config.hidden_size  # 768

        # Linear Projection: 768 → d_model (Eq. 3.6)
        self.projection = nn.Linear(bert_dim, d_model)
        self.layer_norm = nn.LayerNorm(d_model)

        # Optionally freeze bottom layers for layer-wise LR decay strategy
        if freeze_layers > 0:
            for i, layer in enumerate(self.bert.encoder.layer):
                if i < freeze_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        """
        Args:
            input_ids:      (batch, seq_len)
            attention_mask: (batch, seq_len)
        Returns:
            h_text: (batch, seq_len, d_model)  — projected token embeddings
            pooled: (batch, d_model)           — [CLS] token representation
        """
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        # outputs.last_hidden_state: (B, N+2, 768)
        hidden = outputs.last_hidden_state

        # Linear Projection + LayerNorm (Eq. 3.6)
        h_text = self.layer_norm(self.projection(hidden))  # (B, N+2, d_model)
        pooled = h_text[:, 0, :]  # [CLS] token

        return h_text, pooled

    def get_param_groups(self, base_lr: float):
        """
        Layer-wise Learning Rate Decay (Section 3.4):
        - Bottom layers (1-6): lr = base_lr / 10
        - Top layers (7-12) + projection: lr = base_lr
        """
        bottom_params, top_params = [], []
        for i, layer in enumerate(self.bert.encoder.layer):
            if i < 6:
                bottom_params.extend(layer.parameters())
            else:
                top_params.extend(layer.parameters())

        return [
            {'params': bottom_params, 'lr': base_lr / 10},
            {'params': top_params, 'lr': base_lr},
            {'params': self.projection.parameters(), 'lr': base_lr},
            {'params': self.layer_norm.parameters(), 'lr': base_lr},
        ]

### 3.4 Video Encoder: Vision Transformer + Pose Estimation (Section 3.5)

Dual-path video feature extraction for CMDC:
- **Path 1**: ViT-base for facial dynamics (768-dim → d_model)
- **Path 2**: Pose Estimation for body motion features

Both paths go through **Gated Residual** layers, then combined via **Multi-view Gating**.

For D-Vlog: Uses pre-extracted 136-dim OpenFace features via IntraModalFusion.

In [ ]:
class GatedResidual(nn.Module):
    """
    Gated Residual Layer (Section 3.5).
    Inspired by Highway Networks:
        g_res = σ(W_gate · x + b_gate)
        output = g_res ⊙ FFN(x) + (1 - g_res) ⊙ x

    Allows the network to adaptively decide whether to refine or preserve features.
    """
    def __init__(self, d_model: int, dropout: float = 0.3):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model)
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        g = self.gate(x)                        # (B, d_model)
        refined = self.ffn(x)                    # (B, d_model)
        output = g * refined + (1 - g) * x       # Gated residual
        return self.norm(output)


class MultiViewGating(nn.Module):
    """
    Multi-view Gating Layer (Section 3.5).
    Combines face dynamics and body motion features via Softmax soft-attention.

    Output: h_video ∈ R^{d_model=256}
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 2)  # 2 views: face + pose
        )

    def forward(self, face_feat, pose_feat):
        """
        Args:
            face_feat: (batch, d_model)  — ViT face features
            pose_feat: (batch, d_model)  — Pose estimation features
        Returns:
            h_video: (batch, d_model) — fused video representation
        """
        combined = torch.cat([face_feat, pose_feat], dim=-1)  # (B, 2*d_model)
        weights = F.softmax(self.attention(combined), dim=-1)  # (B, 2)

        # Weighted combination
        h_video = weights[:, 0:1] * face_feat + weights[:, 1:2] * pose_feat
        return h_video


class VideoEncoder(nn.Module):
    """
    Video Feature Extractor (Section 3.5).

    For CMDC (trimodal):
        - Path 1: ViT-base face encoder → GatedResidual
        - Path 2: Pose feature encoder → GatedResidual
        - Multi-view Gating → h_video ∈ R^{d_model}

    For D-Vlog (bimodal):
        Uses pre-extracted 136-dim OpenFace features directly through IntraModalFusion.
        In this case, instantiate IntraModalFusion(input_dim=136) instead.

    Args:
        face_dim:  Input dimension for face features
        pose_dim:  Input dimension for pose features
        d_model:   Output embedding dimension (256)
    """
    def __init__(self, face_dim: int, pose_dim: int, d_model: int = 256,
                 dropout: float = 0.3):
        super().__init__()
        # Face path
        self.face_proj = nn.Linear(face_dim, d_model)
        self.face_norm = nn.LayerNorm(d_model)
        self.face_gated_residual = GatedResidual(d_model, dropout)

        # Pose path
        self.pose_proj = nn.Linear(pose_dim, d_model)
        self.pose_norm = nn.LayerNorm(d_model)
        self.pose_gated_residual = GatedResidual(d_model, dropout)

        # Multi-view Gating
        self.multi_view_gating = MultiViewGating(d_model)

    def forward(self, face_feat, pose_feat):
        """
        Args:
            face_feat: (batch, face_dim)
            pose_feat: (batch, pose_dim)
        Returns:
            h_video: (batch, d_model)
        """
        face = self.face_gated_residual(self.face_norm(self.face_proj(face_feat)))
        pose = self.pose_gated_residual(self.pose_norm(self.pose_proj(pose_feat)))
        h_video = self.multi_view_gating(face, pose)
        return h_video

## 4. Multi-Modal Fusion Modules

### 4.1 Bimodal Fusion: Cross-Modal Attention A + Gating g (Section 3.6)

The bimodal fusion module implements:
1. **Cross-Modal Attention Matrix A** (Eq. 3.7): Captures cross-modal semantic correlations
2. **Bidirectional Augmented Representations** (Eq. 3.8): h_audio^aug and h_text^aug
3. **Adaptive Gating Vector g** (Eq. 3.9-3.10): Dimension-level modality contribution weighting

In [ ]:
class CrossModalAttention(nn.Module):
    """
    Cross-Modal Attention Matrix A (Section 3.6.1).

    Computes bilinear attention between two modality representations:
        A = Softmax(h_audio · W_A · h_text^T / √d_k)    (Eq. 3.7)

    Then produces bidirectional augmented representations:
        h_audio^aug = A · h_text                          (Eq. 3.8)
        h_text^aug  = A^T · h_audio                       (Eq. 3.8)

    The learnable projection W_A creates a 'Cross-modal Semantic Bridge Space'
    where pathology-related audio dimensions are aligned with corresponding text dimensions.
    """
    def __init__(self, d_model: int = 256, num_heads: int = 8, dropout: float = 0.3):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        # Learnable projection matrices for Q, K, V
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V_a = nn.Linear(d_model, d_model)  # Value projection for modality A
        self.W_V_b = nn.Linear(d_model, d_model)  # Value projection for modality B

        self.dropout = nn.Dropout(dropout)
        self.norm_a = nn.LayerNorm(d_model)
        self.norm_b = nn.LayerNorm(d_model)

    def forward(self, h_a, h_b):
        """
        Bidirectional Cross-Modal Attention.

        Args:
            h_a: (batch, d_model) — pooled representation of modality A (e.g., audio)
            h_b: (batch, d_model) — pooled representation of modality B (e.g., text)
        Returns:
            h_a_aug: (batch, d_model) — text-augmented audio representation
            h_b_aug: (batch, d_model) — audio-augmented text representation
        """
        # Expand to (B, 1, d_model) for attention computation
        h_a_exp = h_a.unsqueeze(1)  # (B, 1, d)
        h_b_exp = h_b.unsqueeze(1)  # (B, 1, d)

        # A→B attention: audio queries, text keys/values
        Q_a = self.W_Q(h_a_exp)      # (B, 1, d)
        K_b = self.W_K(h_b_exp)      # (B, 1, d)
        V_b = self.W_V_b(h_b_exp)    # (B, 1, d)

        # Scaled dot-product attention: A = Softmax(Q·K^T / √d_k)
        attn_scores = torch.matmul(Q_a, K_b.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        h_a_aug = torch.matmul(attn_weights, V_b).squeeze(1)  # (B, d)

        # B→A attention: text queries, audio keys/values
        Q_b = self.W_Q(h_b_exp)
        K_a = self.W_K(h_a_exp)
        V_a = self.W_V_a(h_a_exp)

        attn_scores_rev = torch.matmul(Q_b, K_a.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights_rev = F.softmax(attn_scores_rev, dim=-1)
        attn_weights_rev = self.dropout(attn_weights_rev)
        h_b_aug = torch.matmul(attn_weights_rev, V_a).squeeze(1)  # (B, d)

        # Residual + LayerNorm
        h_a_aug = self.norm_a(h_a + h_a_aug)
        h_b_aug = self.norm_b(h_b + h_b_aug)

        return h_a_aug, h_b_aug


class GatingFusion(nn.Module):
    """
    Adaptive Gating Vector g (Section 3.6.2).

    Generates dimension-level modality contribution weights:
        g = σ(W_g · [pool(h_a^aug); pool(h_b^aug)] + b_g)   (Eq. 3.9)
        h_fused = g ⊙ h_a^aug + (1-g) ⊙ h_b^aug             (Eq. 3.10)

    Each dimension independently decides audio (g→1) vs text (g→0) contribution,
    handling heterogeneity between 'verbal defense' and 'acoustic pathology' patterns.
    """
    def __init__(self, d_model: int = 256):
        super().__init__()
        self.gate_network = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

    def forward(self, h_a_aug, h_b_aug):
        """
        Args:
            h_a_aug: (batch, d_model) — augmented modality A
            h_b_aug: (batch, d_model) — augmented modality B
        Returns:
            h_fused: (batch, d_model) — gated fusion output
            g:       (batch, d_model) — gating weights (for interpretability)
        """
        concat = torch.cat([h_a_aug, h_b_aug], dim=-1)  # (B, 2*d_model)
        g = self.gate_network(concat)                     # (B, d_model), values in [0,1]

        # Dimension-level weighted combination (Eq. 3.10)
        h_fused = g * h_a_aug + (1 - g) * h_b_aug        # (B, d_model)

        return h_fused, g


class BimodalFusionModule(nn.Module):
    """
    Complete Bimodal Fusion: Cross-Modal Attention A + Gating g (Section 3.6).

    Used by:
        - D-Vlog: Audio + Video bimodal path
        - EATD:   Audio + Text bimodal path (with Word-Level Forced Alignment)
    """
    def __init__(self, d_model: int = 256, num_heads: int = 8, dropout: float = 0.3):
        super().__init__()
        self.cross_attention = CrossModalAttention(d_model, num_heads, dropout)
        self.gating = GatingFusion(d_model)

    def forward(self, h_a, h_b):
        """
        Args:
            h_a, h_b: (batch, d_model) each — pooled modality representations
        Returns:
            h_fused: (batch, d_model)
            g:       (batch, d_model) — gating weights
        """
        h_a_aug, h_b_aug = self.cross_attention(h_a, h_b)
        h_fused, g = self.gating(h_a_aug, h_b_aug)
        return h_fused, g

### 4.2 Trimodal Fusion: Hierarchical G(A,T)→CA(·,V) (Section 3.7)

For CMDC's three-modal scenario, the optimal fusion order (verified by ablation in Table 4-7):

**Stage 1**: G(A,T) — Audio-Text gating fusion first (linguistic conjugation priority)  
**Stage 2**: CA(h_AT, V) — Cross-attention with video as 'visual verifier'  
**Tri-Modal Gate**: Softmax-based w_A + w_T + w_V = 1 (for interpretability)

In [ ]:
class TriModalFusion(nn.Module):
    """
    Hierarchical Tri-Modal Fusion Network: G(A,T) → CA(·,V) (Section 3.7).

    Stage 1: G(A,T) — Gating fusion of audio and text (Eq. 3.11)
             Reuses BimodalFusionModule to produce h_AT ∈ R^{d_model}

    Stage 2: CA(h_AT, V) — Cross-modal attention with video (Eq. 3.12-3.13)
             h_AT as Query, h_video as Key/Value
             h_ATV = h_AT + M_{AT→V} · (h_video · W_V)

    Tri-Modal Gate: Softmax soft-attention (Eq. 3.14-3.15)
             [w_A, w_T, w_V] = Softmax(MLP([pool_A; pool_T; pool_V]))
             h_tri = w_A · pool_A + w_T · pool_T + w_V · pool_V

    Args:
        d_model: Input embedding dimension (256)
    """
    def __init__(self, d_model: int = 256, num_heads: int = 8, dropout: float = 0.3):
        super().__init__()
        self.d_model = d_model

        # Stage 1: G(A,T) — Bimodal gating fusion (reuse)
        self.stage1_gating = BimodalFusionModule(d_model, num_heads, dropout)

        # Stage 2: Cross-Attention with Video
        # h_AT as Query, h_video as Key/Value
        self.stage2_W_Q = nn.Linear(d_model, d_model)
        self.stage2_W_K = nn.Linear(d_model, d_model)
        self.stage2_W_V = nn.Linear(d_model, d_model)
        self.stage2_norm = nn.LayerNorm(d_model)
        self.stage2_dropout = nn.Dropout(dropout)

        # Projection: d_model (256) → 512 for Self-Modal Attention
        self.project_up = nn.Linear(d_model, 512)

        # Self-Modal Attention (d=512) for high-order feature interaction
        self.self_modal_attention = nn.TransformerEncoderLayer(
            d_model=512,
            nhead=num_heads,
            dim_feedforward=2048,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )

        # Tri-Modal Gate: MLP → Softmax over 3 modalities (Eq. 3.14)
        self.tri_gate_mlp = nn.Sequential(
            nn.Linear(512 * 3, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 3)  # 3 weights: w_A, w_T, w_V
        )

        # Individual modality projections to 512-dim for Tri-Modal Gate
        self.proj_a = nn.Linear(d_model, 512)
        self.proj_t = nn.Linear(d_model, 512)
        self.proj_v = nn.Linear(d_model, 512)

    def forward(self, h_audio, h_text, h_video):
        """
        Args:
            h_audio: (batch, d_model) — audio encoder output
            h_text:  (batch, d_model) — text encoder output
            h_video: (batch, d_model) — video encoder output
        Returns:
            h_tri:       (batch, 512)  — final trimodal representation
            weights:     (batch, 3)    — [w_A, w_T, w_V] contribution weights
            g_stage1:    (batch, d_model) — Stage 1 gating weights
        """
        # ---- Stage 1: G(A,T) ---- (Eq. 3.11)
        h_AT, g_stage1 = self.stage1_gating(h_audio, h_text)  # (B, d_model)

        # ---- Stage 2: CA(h_AT, V) ---- (Eq. 3.12-3.13)
        # h_AT as Query, h_video as Key/Value
        Q = self.stage2_W_Q(h_AT).unsqueeze(1)       # (B, 1, d)
        K = self.stage2_W_K(h_video).unsqueeze(1)     # (B, 1, d)
        V = self.stage2_W_V(h_video).unsqueeze(1)     # (B, 1, d)

        d_k = self.d_model // 8
        attn = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, 1, 1)
        attn = F.softmax(attn, dim=-1)
        attn = self.stage2_dropout(attn)
        h_ATV = h_AT + torch.matmul(attn, V).squeeze(1)  # Residual: (B, d_model)
        h_ATV = self.stage2_norm(h_ATV)

        # ---- Project to 512-dim ----
        h_ATV_up = self.project_up(h_ATV).unsqueeze(1)  # (B, 1, 512)
        h_ATV_up = self.self_modal_attention(h_ATV_up).squeeze(1)  # (B, 512)

        # ---- Tri-Modal Gate ---- (Eq. 3.14-3.15)
        pool_A = self.proj_a(h_audio)  # (B, 512)
        pool_T = self.proj_t(h_text)   # (B, 512)
        pool_V = self.proj_v(h_video)  # (B, 512)

        gate_input = torch.cat([pool_A, pool_T, pool_V], dim=-1)  # (B, 1536)
        weights = F.softmax(self.tri_gate_mlp(gate_input), dim=-1)  # (B, 3)

        # Weighted combination (Eq. 3.15)
        w_A = weights[:, 0:1]  # (B, 1)
        w_T = weights[:, 1:2]
        w_V = weights[:, 2:3]
        h_tri = w_A * pool_A + w_T * pool_T + w_V * pool_V  # (B, 512)

        return h_tri, weights, g_stage1

### 4.3 Classification Head & Loss Function (Section 3.8)

- **Classifier**: 512 → 256 → 128 → 2, with LayerNorm + Dropout(0.3) between layers
- **Loss**: Weighted Cross-Entropy (Eq. 3.16): w_c = N_total / (C × N_c)

In [ ]:
class Classifier(nn.Module):
    """
    Three-layer Classification Head (Section 3.8).
    Architecture: 512 → 256 → 128 → 2
    Each layer: Linear → LayerNorm → GELU → Dropout

    Note: Bimodal path first projects d_model(256) → 512, then shares
    the same classifier topology as the trimodal path.
    """
    def __init__(self, input_dim: int = 512, dropout: float = 0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 2)  # Binary classification: Depression / Non-depression
        )

    def forward(self, x):
        return self.classifier(x)


def compute_class_weights(labels, num_classes=2):
    """
    Compute inverse-frequency class weights (Eq. 3.16).
    w_c = N_total / (C × N_c)

    Example (D-Vlog): 307 depressed + 500 healthy = 807 total
        w_dep = 807 / (2 × 307) ≈ 1.314
        w_nor = 807 / (2 × 500) ≈ 0.807
    """
    labels = np.array(labels)
    N_total = len(labels)
    weights = []
    for c in range(num_classes):
        N_c = (labels == c).sum()
        w_c = N_total / (num_classes * max(N_c, 1))
        weights.append(w_c)
    return torch.FloatTensor(weights)

## 5. Complete Detection Models for Each Dataset

### 5.1 D-Vlog Model: Audio(25-dim) + Video(136-dim) Bimodal

In [ ]:
class DVlogDepressionDetector(nn.Module):
    """
    IEMDDM for D-Vlog Dataset (Section 4.1.1).

    Modality: Audio (25-dim pre-extracted features) + Video (136-dim OpenFace AU features)
    Fusion:   Cross-Modal Attention A + Gating g

    Pipeline:
        Audio → IntraModalFusion(25→256) → h_audio
        Video → IntraModalFusion(136→256) → h_video
        [h_audio, h_video] → BimodalFusion(A+g) → h_fused(256)
        h_fused → Linear(256→512) → Classifier(512→256→128→2)
    """
    def __init__(self, audio_dim: int = 25, video_dim: int = 136,
                 d_model: int = 256, num_heads: int = 8, dropout: float = 0.3):
        super().__init__()

        # Modality Encoders (shared IntraModalFusion architecture)
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)
        self.video_encoder = IntraModalFusion(video_dim, d_model, num_heads, dropout)

        # Bimodal Fusion: Cross-Attention A + Gating g
        self.fusion = BimodalFusionModule(d_model, num_heads, dropout)

        # Projection: 256 → 512 (to match classifier input)
        self.project_up = nn.Linear(d_model, 512)
        self.proj_norm = nn.LayerNorm(512)

        # Classification Head: 512 → 256 → 128 → 2
        self.classifier = Classifier(512, dropout)

    def forward(self, audio, video):
        """
        Args:
            audio: (batch, T_A, 25)   — audio feature sequence
            video: (batch, T_V, 136)  — video feature sequence
        Returns:
            logits: (batch, 2)
        """
        _, h_audio = self.audio_encoder(audio)  # (B, 256)
        _, h_video = self.video_encoder(video)  # (B, 256)

        h_fused, g = self.fusion(h_audio, h_video)  # (B, 256)

        h_up = self.proj_norm(self.project_up(h_fused))  # (B, 512)
        logits = self.classifier(h_up)  # (B, 2)

        return logits

### 5.2 EATD Model: Audio + Text Bimodal (with Word-Level Forced Alignment)

In [ ]:
class EATDDepressionDetector(nn.Module):
    """
    IEMDDM for EATD Dataset (Section 4.1.2).

    Modality: Audio (MFA word-level aligned acoustic features) + Text (bert-base-chinese)
    Fusion:   Cross-Modal Attention A + Gating g
    Pre-processing: Word-Level Forced Alignment via MFA (Section 3.2)

    Pipeline:
        Audio → IntraModalFusion(audio_dim→256) → h_audio
        Text  → bert-base-chinese → Linear(768→256) → h_text   [OR pre-extracted 768-dim features]
        [h_audio, h_text] → BimodalFusion(A+g) → h_fused(256)
        h_fused → Linear(256→512) → Classifier(512→256→128→2)

    Note: If using pre-extracted BERT features (768-dim vectors), set use_pretrained_bert=False
    and provide audio_dim and text_dim accordingly.
    """
    def __init__(self, audio_dim: int, text_dim: int = 768,
                 d_model: int = 256, num_heads: int = 8, dropout: float = 0.3,
                 use_pretrained_bert: bool = False):
        super().__init__()
        self.use_pretrained_bert = use_pretrained_bert

        # Audio Encoder
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)

        # Text Encoder
        if use_pretrained_bert:
            self.text_encoder = TextEncoder('bert-base-chinese', d_model)
        else:
            # For pre-extracted text features (e.g., 768-dim BERT [CLS] embeddings)
            self.text_proj = nn.Linear(text_dim, d_model)
            self.text_norm = nn.LayerNorm(d_model)

        # Bimodal Fusion
        self.fusion = BimodalFusionModule(d_model, num_heads, dropout)

        # Projection + Classifier
        self.project_up = nn.Linear(d_model, 512)
        self.proj_norm = nn.LayerNorm(512)
        self.classifier = Classifier(512, dropout)

    def forward(self, audio_feat, text_feat, text_input_ids=None, text_attention_mask=None):
        """
        Args:
            audio_feat: (batch, T_A, audio_dim) or (batch, audio_dim) — audio features
            text_feat:  (batch, text_dim)        — pre-extracted BERT features
                    OR  None if using live BERT encoding
            text_input_ids, text_attention_mask: for live BERT encoding
        """
        # Handle audio input shape
        if audio_feat.dim() == 2:
            audio_feat = audio_feat.unsqueeze(1)  # (B, 1, audio_dim)
        _, h_audio = self.audio_encoder(audio_feat)

        # Text encoding
        if self.use_pretrained_bert and text_input_ids is not None:
            _, h_text = self.text_encoder(text_input_ids, text_attention_mask)
        else:
            h_text = self.text_norm(self.text_proj(text_feat))

        # Fusion
        h_fused, g = self.fusion(h_audio, h_text)

        # Classify
        h_up = self.proj_norm(self.project_up(h_fused))
        logits = self.classifier(h_up)

        return logits

### 5.3 CMDC Model: Audio + Text + Video Trimodal with G(A,T)→CA(·,V)

In [ ]:
class CMDCDepressionDetector(nn.Module):
    """
    IEMDDM for CMDC Dataset (Section 4.1.3).

    Modality: Audio + Text (bert-base-chinese) + Video (ViT + Pose)
    Fusion:   Hierarchical G(A,T) → CA(·,V) + Tri-Modal Gate

    Pipeline:
        Audio → IntraModalFusion → h_audio (256)
        Text  → bert-base-chinese → Linear(768→256) → h_text (256)
        Video → VideoEncoder(ViT+Pose→GatedResidual→MultiViewGating) → h_video (256)

        Stage 1: G(A,T) → h_AT                          (Eq. 3.11)
        Stage 2: CA(h_AT, V) → h_ATV                    (Eq. 3.12-3.13)
        Tri-Modal Gate: Softmax(w_A, w_T, w_V) → h_tri  (Eq. 3.14-3.15)

        h_tri(512) → Classifier(512→256→128→2)
    """
    def __init__(self, audio_dim: int, text_dim: int = 768,
                 face_dim: int = 768, pose_dim: int = 50,
                 d_model: int = 256, num_heads: int = 8, dropout: float = 0.3,
                 use_pretrained_bert: bool = False):
        super().__init__()
        self.use_pretrained_bert = use_pretrained_bert

        # === Modality Encoders ===
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)

        if use_pretrained_bert:
            self.text_encoder = TextEncoder('bert-base-chinese', d_model)
        else:
            self.text_proj = nn.Linear(text_dim, d_model)
            self.text_norm = nn.LayerNorm(d_model)

        self.video_encoder = VideoEncoder(face_dim, pose_dim, d_model, dropout)

        # === Trimodal Fusion: G(A,T) → CA(·,V) ===
        self.tri_modal_fusion = TriModalFusion(d_model, num_heads, dropout)

        # === Classification Head: 512 → 256 → 128 → 2 ===
        self.classifier = Classifier(512, dropout)

    def forward(self, audio_feat, text_feat, face_feat, pose_feat,
                text_input_ids=None, text_attention_mask=None):
        """
        Args:
            audio_feat: (batch, T_A, audio_dim) or (batch, audio_dim)
            text_feat:  (batch, text_dim) — pre-extracted BERT features
            face_feat:  (batch, face_dim) — ViT face features
            pose_feat:  (batch, pose_dim) — Pose estimation features
        Returns:
            logits:  (batch, 2)
            weights: (batch, 3) — [w_A, w_T, w_V] for interpretability
        """
        # Audio Encoding
        if audio_feat.dim() == 2:
            audio_feat = audio_feat.unsqueeze(1)
        _, h_audio = self.audio_encoder(audio_feat)

        # Text Encoding
        if self.use_pretrained_bert and text_input_ids is not None:
            _, h_text = self.text_encoder(text_input_ids, text_attention_mask)
        else:
            h_text = self.text_norm(self.text_proj(text_feat))

        # Video Encoding (dual-path: face + pose → GatedResidual → MultiViewGating)
        h_video = self.video_encoder(face_feat, pose_feat)

        # Trimodal Fusion: G(A,T) → CA(·,V)
        h_tri, weights, g_stage1 = self.tri_modal_fusion(h_audio, h_text, h_video)

        # Classification
        logits = self.classifier(h_tri)

        return logits, weights

## 6. Ablation Experiment Models (Section 4.4)

Ablation variants for D-Vlog and EATD to decompose module contributions.

In [ ]:
# ==================== D-Vlog Ablation Models ====================

class AudioOnlyModel(nn.Module):
    """Ablation: Audio modality only."""
    def __init__(self, audio_dim=25, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)
        self.project_up = nn.Linear(d_model, 512)
        self.classifier = Classifier(512, dropout)

    def forward(self, audio, video=None):
        _, pooled = self.encoder(audio)
        return self.classifier(self.project_up(pooled))


class VideoOnlyModel(nn.Module):
    """Ablation: Video modality only."""
    def __init__(self, video_dim=136, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.encoder = IntraModalFusion(video_dim, d_model, num_heads, dropout)
        self.project_up = nn.Linear(d_model, 512)
        self.classifier = Classifier(512, dropout)

    def forward(self, audio, video):
        _, pooled = self.encoder(video)
        return self.classifier(self.project_up(pooled))


class EarlyFusionModel(nn.Module):
    """Ablation: Early fusion (feature concatenation)."""
    def __init__(self, audio_dim=25, video_dim=136, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.encoder = IntraModalFusion(audio_dim + video_dim, d_model, num_heads, dropout)
        self.project_up = nn.Linear(d_model, 512)
        self.classifier = Classifier(512, dropout)

    def forward(self, audio, video):
        combined = torch.cat([audio, video], dim=-1)
        _, pooled = self.encoder(combined)
        return self.classifier(self.project_up(pooled))


class LateFusionModel(nn.Module):
    """Ablation: Late fusion (decision-level weighted average)."""
    def __init__(self, audio_dim=25, video_dim=136, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)
        self.video_encoder = IntraModalFusion(video_dim, d_model, num_heads, dropout)
        self.audio_classifier = Classifier(512, dropout)
        self.video_classifier = Classifier(512, dropout)
        self.project_a = nn.Linear(d_model, 512)
        self.project_v = nn.Linear(d_model, 512)
        self.fusion_weight = nn.Parameter(torch.tensor(0.5))

    def forward(self, audio, video):
        _, h_a = self.audio_encoder(audio)
        _, h_v = self.video_encoder(video)
        out_a = self.audio_classifier(self.project_a(h_a))
        out_v = self.video_classifier(self.project_v(h_v))
        w = torch.sigmoid(self.fusion_weight)
        return w * out_a + (1 - w) * out_v


class ConcatFusionModel(nn.Module):
    """Ablation: Concatenation fusion (no attention or gating)."""
    def __init__(self, audio_dim=25, video_dim=136, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)
        self.video_encoder = IntraModalFusion(video_dim, d_model, num_heads, dropout)
        self.classifier = Classifier(d_model * 2, dropout)

    def forward(self, audio, video):
        _, h_a = self.audio_encoder(audio)
        _, h_v = self.video_encoder(video)
        fused = torch.cat([h_a, h_v], dim=-1)
        return self.classifier(fused)


class NoCrossAttentionModel(nn.Module):
    """Ablation: w/o Cross-Attention A (gating only)."""
    def __init__(self, audio_dim=25, video_dim=136, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)
        self.video_encoder = IntraModalFusion(video_dim, d_model, num_heads, dropout)
        self.gating = GatingFusion(d_model)
        self.project_up = nn.Linear(d_model, 512)
        self.proj_norm = nn.LayerNorm(512)
        self.classifier = Classifier(512, dropout)

    def forward(self, audio, video):
        _, h_a = self.audio_encoder(audio)
        _, h_v = self.video_encoder(video)
        h_fused, _ = self.gating(h_a, h_v)
        return self.classifier(self.proj_norm(self.project_up(h_fused)))


class NoGatingModel(nn.Module):
    """Ablation: w/o Gating g (cross-attention only, average pooling)."""
    def __init__(self, audio_dim=25, video_dim=136, d_model=256, num_heads=8, dropout=0.3):
        super().__init__()
        self.audio_encoder = IntraModalFusion(audio_dim, d_model, num_heads, dropout)
        self.video_encoder = IntraModalFusion(video_dim, d_model, num_heads, dropout)
        self.cross_attn = CrossModalAttention(d_model, num_heads, dropout)
        self.project_up = nn.Linear(d_model, 512)
        self.proj_norm = nn.LayerNorm(512)
        self.classifier = Classifier(512, dropout)

    def forward(self, audio, video):
        _, h_a = self.audio_encoder(audio)
        _, h_v = self.video_encoder(video)
        h_a_aug, h_v_aug = self.cross_attn(h_a, h_v)
        h_fused = (h_a_aug + h_v_aug) / 2  # Simple average instead of gating
        return self.classifier(self.proj_norm(self.project_up(h_fused)))

## 7. CMDC Trimodal Fusion Order Ablation (Section 4.5, Table 4-7)

Six fusion order variants for systematic evaluation:

| Order | Stage 1 | Stage 2 | F1 (reported) |
|-------|---------|---------|---------------|
| **★ G(A,T)→CA(·,V)** | Gating(Audio, Text) | CrossAttn(·, Video) | **0.7544** |
| CA(T,V)→G(·,A) | CrossAttn(Text, Video) | Gating(·, Audio) | 0.7270 |
| G(A,V)→CA(·,T) | Gating(Audio, Video) | CrossAttn(·, Text) | 0.7121 |
| CA(A,T)→G(·,V) | CrossAttn(Audio, Text) | Gating(·, Video) | 0.7187 |
| G(T,V)→CA(·,A) | Gating(Text, Video) | CrossAttn(·, Audio) | 0.7027 |
| CA(A,V)→G(·,T) | CrossAttn(Audio, Video) | Gating(·, Text) | 0.7098 |

In [ ]:
class TriModalAblation(nn.Module):
    """
    Generalized Tri-Modal Fusion with configurable fusion order.

    This module allows testing all 6 fusion order permutations from Table 4-7.

    Args:
        d_model: Embedding dimension (256)
        first_op: 'gating' or 'cross_attn' — operator for Stage 1
        first_pair: tuple of 2 modality names for Stage 1 (e.g., ('audio', 'text'))
        second_modality: remaining modality name for Stage 2
    """
    def __init__(self, d_model=256, num_heads=8, dropout=0.3,
                 first_op='gating', first_pair=('audio', 'text'),
                 second_modality='video'):
        super().__init__()
        self.first_op = first_op
        self.first_pair = first_pair
        self.second_modality = second_modality

        # Stage 1 operator
        if first_op == 'gating':
            self.stage1 = BimodalFusionModule(d_model, num_heads, dropout)
        else:  # cross_attn
            self.stage1_cross = CrossModalAttention(d_model, num_heads, dropout)

        # Stage 2 operator (opposite of stage 1)
        if first_op == 'gating':
            # Stage 2 uses cross-attention
            self.stage2_W_Q = nn.Linear(d_model, d_model)
            self.stage2_W_K = nn.Linear(d_model, d_model)
            self.stage2_W_V = nn.Linear(d_model, d_model)
            self.stage2_norm = nn.LayerNorm(d_model)
        else:
            # Stage 2 uses gating
            self.stage2_gating = GatingFusion(d_model)

        self.project_up = nn.Linear(d_model, 512)
        self.proj_norm = nn.LayerNorm(512)
        self.classifier = Classifier(512, dropout)
        self.stage2_dropout = nn.Dropout(dropout)

    def _get_modality(self, modalities, name):
        return modalities[name]

    def forward(self, h_audio, h_text, h_video):
        modalities = {'audio': h_audio, 'text': h_text, 'video': h_video}

        m1 = self._get_modality(modalities, self.first_pair[0])
        m2 = self._get_modality(modalities, self.first_pair[1])
        m3 = self._get_modality(modalities, self.second_modality)

        # Stage 1
        if self.first_op == 'gating':
            h_12, _ = self.stage1(m1, m2)
        else:
            h_12_a, h_12_b = self.stage1_cross(m1, m2)
            h_12 = (h_12_a + h_12_b) / 2

        # Stage 2
        if self.first_op == 'gating':
            # Cross-attention with third modality
            Q = self.stage2_W_Q(h_12).unsqueeze(1)
            K = self.stage2_W_K(m3).unsqueeze(1)
            V = self.stage2_W_V(m3).unsqueeze(1)
            attn = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(32)
            attn = F.softmax(attn, dim=-1)
            attn = self.stage2_dropout(attn)
            h_final = h_12 + torch.matmul(attn, V).squeeze(1)
            h_final = self.stage2_norm(h_final)
        else:
            h_final, _ = self.stage2_gating(h_12, m3)

        h_up = self.proj_norm(self.project_up(h_final))
        logits = self.classifier(h_up)
        return logits


# Define all 6 fusion order configurations
FUSION_ORDERS = {
    'G(A,T)->CA(.,V)': {'first_op': 'gating',     'first_pair': ('audio', 'text'),  'second_modality': 'video'},
    'CA(T,V)->G(.,A)': {'first_op': 'cross_attn', 'first_pair': ('text', 'video'),  'second_modality': 'audio'},
    'G(A,V)->CA(.,T)': {'first_op': 'gating',     'first_pair': ('audio', 'video'), 'second_modality': 'text'},
    'CA(A,T)->G(.,V)': {'first_op': 'cross_attn', 'first_pair': ('audio', 'text'),  'second_modality': 'video'},
    'G(T,V)->CA(.,A)': {'first_op': 'gating',     'first_pair': ('text', 'video'),  'second_modality': 'audio'},
    'CA(A,V)->G(.,T)': {'first_op': 'cross_attn', 'first_pair': ('audio', 'video'), 'second_modality': 'text'},
}

## 8. Training & Evaluation Pipeline

Unified training loop with:
- AdamW optimizer (lr=1e-4, weight_decay=1e-4)
- Cosine Annealing LR scheduler
- Early stopping (patience=10, monitor: val F1-Score)
- Weighted Cross-Entropy Loss

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, dataset_type='dvlog'):
    """Single training epoch."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in dataloader:
        if dataset_type == 'dvlog':
            audio = batch['audio'].to(device)
            video = batch['video'].to(device)
            labels = batch['label'].to(device)
            outputs = model(audio, video)
        elif dataset_type == 'eatd':
            audio = batch['audio'].to(device)
            text = batch['text'].to(device)
            labels = batch['label'].to(device)
            outputs = model(audio, text)
        elif dataset_type == 'cmdc':
            audio = batch['audio'].to(device)
            text = batch['text'].to(device)
            face = batch['face'].to(device)
            pose = batch['pose'].to(device)
            labels = batch['label'].to(device)
            outputs = model(audio, text, face, pose)
            if isinstance(outputs, tuple):
                outputs = outputs[0]  # logits only

        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    return avg_loss


@torch.no_grad()
def evaluate(model, dataloader, device, dataset_type='dvlog'):
    """Evaluate model and return metrics dictionary."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for batch in dataloader:
        if dataset_type == 'dvlog':
            audio = batch['audio'].to(device)
            video = batch['video'].to(device)
            labels = batch['label'].to(device)
            outputs = model(audio, video)
        elif dataset_type == 'eatd':
            audio = batch['audio'].to(device)
            text = batch['text'].to(device)
            labels = batch['label'].to(device)
            outputs = model(audio, text)
        elif dataset_type == 'cmdc':
            audio = batch['audio'].to(device)
            text = batch['text'].to(device)
            face = batch['face'].to(device)
            pose = batch['pose'].to(device)
            labels = batch['label'].to(device)
            outputs = model(audio, text, face, pose)
            if isinstance(outputs, tuple):
                outputs = outputs[0]

        probs = F.softmax(outputs, dim=1)[:, 1]
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    metrics = {
        'accuracy':  accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall':    recall_score(all_labels, all_preds, zero_division=0),
        'f1':        f1_score(all_labels, all_preds, zero_division=0),
    }
    try:
        metrics['auc'] = roc_auc_score(all_labels, all_probs)
    except ValueError:
        metrics['auc'] = 0.0

    return metrics


def train_model(model, train_loader, val_loader, test_loader, config, device,
                dataset_type='dvlog', class_weights=None, verbose=True):
    """
    Full training pipeline with early stopping and cosine annealing.

    Args:
        model: nn.Module
        train_loader, val_loader, test_loader: DataLoaders
        config: Config dataclass
        device: torch.device
        dataset_type: 'dvlog', 'eatd', or 'cmdc'
        class_weights: tensor of shape (2,) for Weighted Cross-Entropy
    Returns:
        test_metrics: dict with accuracy, precision, recall, f1, auc
    """
    model = model.to(device)

    # Weighted Cross-Entropy Loss (Eq. 3.16)
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.max_epochs
    )

    best_val_f1 = 0
    best_state = None
    patience_counter = 0

    for epoch in range(config.max_epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, dataset_type)
        val_metrics = evaluate(model, val_loader, device, dataset_type)
        scheduler.step()

        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {train_loss:.4f} | "
                  f"Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config.patience:
            if verbose:
                print(f"  Early stopping at epoch {epoch+1}")
            break

    # Load best model and evaluate on test set
    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate(model, test_loader, device, dataset_type)
    return test_metrics

## 9. Dataset Classes

Generic dataset wrappers for each benchmark. Replace data loading logic  
with your actual data paths and preprocessing pipeline.

In [ ]:
class DVlogDataset(Dataset):
    """
    D-Vlog Dataset (Section 4.1.1).
    Audio: 25-dim pre-extracted features, shape (T, 25)
    Video: 136-dim OpenFace features, shape (T, 136)
    Label: 0 (healthy) / 1 (depressed), PHQ-8 ≥ 10

    Data split: Official train(569)/val(119)/test(119)
    """
    def __init__(self, audio_features, video_features, labels, max_len=300):
        self.audio = audio_features   # list of (T_i, 25) arrays
        self.video = video_features   # list of (T_i, 136) arrays
        self.labels = labels          # list of int
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def _pad_or_truncate(self, feat, max_len):
        if len(feat) >= max_len:
            return feat[:max_len]
        else:
            pad = np.zeros((max_len - len(feat), feat.shape[1]))
            return np.concatenate([feat, pad], axis=0)

    def __getitem__(self, idx):
        audio = self._pad_or_truncate(self.audio[idx], self.max_len)
        video = self._pad_or_truncate(self.video[idx], self.max_len)
        return {
            'audio': torch.FloatTensor(audio),
            'video': torch.FloatTensor(video),
            'label': torch.LongTensor([self.labels[idx]])[0]
        }


class EATDDataset(Dataset):
    """
    EATD Dataset (Section 4.1.2).
    Audio: Pre-extracted acoustic features (e.g., 128-dim mel features × 3 conditions)
    Text:  Pre-extracted BERT [CLS] embeddings (768-dim × 3 conditions)
    Label: 0 (healthy, SDS<53) / 1 (depressed, SDS≥53)

    Evaluation: 5-fold stratified cross-validation
    """
    def __init__(self, audio_features, text_features, labels):
        self.audio = audio_features  # (N, audio_dim)
        self.text = text_features    # (N, text_dim)
        self.labels = labels         # (N,)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'audio': torch.FloatTensor(self.audio[idx]),
            'text':  torch.FloatTensor(self.text[idx]),
            'label': torch.LongTensor([self.labels[idx]])[0]
        }


class CMDCDataset(Dataset):
    """
    CMDC Dataset (Section 4.1.3).
    Audio: Pre-extracted acoustic features
    Text:  Pre-extracted BERT [CLS] embeddings
    Face:  Pre-extracted ViT/ResNet face features
    Pose:  Pre-extracted pose estimation features
    Label: 0 (healthy) / 1 (depressed), PHQ-9 + HAMD-17 confirmed

    Evaluation: 5-fold stratified cross-validation
    """
    def __init__(self, audio_features, text_features, face_features,
                 pose_features, labels):
        self.audio = audio_features
        self.text = text_features
        self.face = face_features
        self.pose = pose_features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'audio': torch.FloatTensor(self.audio[idx]),
            'text':  torch.FloatTensor(self.text[idx]),
            'face':  torch.FloatTensor(self.face[idx]),
            'pose':  torch.FloatTensor(self.pose[idx]),
            'label': torch.LongTensor([self.labels[idx]])[0]
        }

## 10. Quick Verification (Synthetic Data)

Run a forward pass with synthetic data to verify all model architectures are correct.

In [ ]:
def verify_models():
    """Verify all models with synthetic data."""
    B = 4  # batch size
    print("=" * 70)
    print("Model Architecture Verification with Synthetic Data")
    print("=" * 70)

    # --- D-Vlog ---
    print("\n[D-Vlog] Audio(25-dim) + Video(136-dim) Bimodal")
    model_dvlog = DVlogDepressionDetector(audio_dim=25, video_dim=136)
    audio = torch.randn(B, 300, 25)   # 300 frames, 25-dim
    video = torch.randn(B, 300, 136)  # 300 frames, 136-dim
    logits = model_dvlog(audio, video)
    params = sum(p.numel() for p in model_dvlog.parameters())
    print(f"  Output shape: {logits.shape}  (expected: [{B}, 2])")
    print(f"  Total parameters: {params:,}")

    # --- EATD ---
    print("\n[EATD] Audio(384-dim) + Text(768-dim) Bimodal (pre-extracted)")
    model_eatd = EATDDepressionDetector(audio_dim=384, text_dim=768*3)
    audio = torch.randn(B, 384)     # pre-extracted audio features
    text = torch.randn(B, 768*3)    # 3 conditions × 768-dim BERT
    logits = model_eatd(audio, text)
    params = sum(p.numel() for p in model_eatd.parameters())
    print(f"  Output shape: {logits.shape}  (expected: [{B}, 2])")
    print(f"  Total parameters: {params:,}")

    # --- CMDC ---
    print("\n[CMDC] Audio + Text + Video Trimodal (pre-extracted)")
    model_cmdc = CMDCDepressionDetector(
        audio_dim=512, text_dim=768, face_dim=768, pose_dim=50
    )
    audio = torch.randn(B, 512)    # pre-extracted audio
    text = torch.randn(B, 768)     # BERT [CLS]
    face = torch.randn(B, 768)     # ViT face features
    pose = torch.randn(B, 50)      # Pose features
    logits, weights = model_cmdc(audio, text, face, pose)
    params = sum(p.numel() for p in model_cmdc.parameters())
    print(f"  Output shape: {logits.shape}  (expected: [{B}, 2])")
    print(f"  Tri-Modal weights shape: {weights.shape}  (expected: [{B}, 3])")
    print(f"  Sample weights (w_A, w_T, w_V): {weights[0].detach().numpy().round(3)}")
    print(f"  Total parameters: {params:,}")

    # --- Ablation ---
    print("\n[Ablation] Fusion Order Variants")
    for name, cfg in FUSION_ORDERS.items():
        model = TriModalAblation(d_model=256, **cfg)
        h_a = torch.randn(B, 256)
        h_t = torch.randn(B, 256)
        h_v = torch.randn(B, 256)
        out = model(h_a, h_t, h_v)
        print(f"  {name:25s} → output shape: {out.shape}")

    print("\n" + "=" * 70)
    print("All models verified successfully!")
    print("=" * 70)

verify_models()

Model Architecture Verification with Synthetic Data

[D-Vlog] Audio(25-dim) + Video(136-dim) Bimodal
  Output shape: torch.Size([4, 2])  (expected: [4, 2])
  Total parameters: 2,315,650

[EATD] Audio(384-dim) + Text(768-dim) Bimodal (pre-extracted)
  Output shape: torch.Size([4, 2])  (expected: [4, 2])
  Total parameters: 2,172,802

[CMDC] Audio + Text + Video Trimodal (pre-extracted)
  Output shape: torch.Size([4, 2])  (expected: [4, 2])
  Tri-Modal weights shape: torch.Size([4, 3])  (expected: [4, 3])
  Sample weights (w_A, w_T, w_V): [0.363 0.283 0.354]
  Total parameters: 7,083,399

[Ablation] Fusion Order Variants
  G(A,T)->CA(.,V)           → output shape: torch.Size([4, 2])
  CA(T,V)->G(.,A)           → output shape: torch.Size([4, 2])
  G(A,V)->CA(.,T)           → output shape: torch.Size([4, 2])
  CA(A,T)->G(.,V)           → output shape: torch.Size([4, 2])
  G(T,V)->CA(.,A)           → output shape: torch.Size([4, 2])
  CA(A,V)->G(.,T)           → output shape: torch.Size([4,